In [5]:
# ============================================================
#   ESO MUSE HYPERSPECTRAL PIPELINE (FOV POLYGON SEARCH)
#   Plate-solve -> parity check -> rotate North-up -> query ESO TAP -> download overlapping cubes
# ============================================================
#
#   Required packages:
#      pip install astropy astroquery pyvo requests pillow numpy scipy tifffile imagecodecs
#
# ============================================================

import argparse
import os
import time
from pathlib import Path

import numpy as np
import pyvo as vo
import requests
import scipy.ndimage
import tifffile
from astropy.io import fits
from astropy.wcs import WCS
from astroquery.astrometry_net import AstrometryNet

# ============================================================
# CONFIGURATION FOR JUPYTER NOTEBOOK / SCRIPT
# ============================================================

# 1. Define your organized working directories
BASE_DIR = Path("./SOLVE")
INPUT_DIR = BASE_DIR / "INPUT"
ALIGNED_DIR = BASE_DIR / "ALIGNED"
CUBE_DIR = BASE_DIR / "MUSE_CUBES"
MANIFEST_DIR = BASE_DIR / "manifests"
TEMP_DIR = BASE_DIR / "temp"

# 2. API Keys and Credentials
ASTROMETRY_API_KEY = os.getenv("ASTROMETRY_API_KEY", "your_actual_astrometry_key_here")
ESO_USERNAME = os.getenv("ESO_USERNAME", None)
ESO_PASSWORD = os.getenv("ESO_PASSWORD", None)

# 3. Payload Customization (For Testing)
MAX_CUBES = 1  # Limit the number of cubes downloaded per field. Set to None to download ALL.


# ──────────────────────────────────────────────────────────────────────────────
#   GLOBAL CONSTANTS
# ──────────────────────────────────────────────────────────────────────────────

ESO_TAP_URL = "https://archive.eso.org/tap_obs"
DOWNSAMPLE_FACTOR = 4
DOWNLOAD_TIMEOUT = (15, 600)

# ──────────────────────────────────────────────────────────────────────────────
#   STEP 1: PRE-FILTER & DOWNSAMPLE
# ──────────────────────────────────────────────────────────────────────────────

def prepare_image_for_solving(img_path: Path, temp_dir: Path, downsample_factor: int = 4) -> Path:
    try:
        with fits.open(img_path) as hdul:
            data   = hdul[0].data.astype(np.float32)
            header = hdul[0].header.copy()

        if data.ndim == 3:
            if data.shape[0] in (3, 4):
                data = np.mean(data, axis=0)
            elif data.shape[-1] in (3, 4):
                data = np.mean(data, axis=-1)
            else:
                data = data[0]
        elif data.ndim > 3:
            while data.ndim > 2:
                data = data[0]

        if downsample_factor > 1:
            data = data[::downsample_factor, ::downsample_factor]

        d_min, d_max = np.nanpercentile(data, 1.0), np.nanpercentile(data, 99.0)
        if d_max > d_min:
            data = np.clip((data - d_min) / (d_max - d_min), 0.0, 1.0) * 65535.0
        else:
            data = np.zeros_like(data)
        data = np.nan_to_num(data, nan=0.0)

        solve_path = temp_dir / f"solve_{img_path.name}"
        fits.PrimaryHDU(data=data, header=header).writeto(
            solve_path, overwrite=True
        )
        return solve_path

    except Exception as e:
        print(f"  [pre-filter] Warning for {img_path.name}: {e}. Using original.")
        return img_path


# ──────────────────────────────────────────────────────────────────────────────
#   STEP 2: PLATE SOLVE WITH RETRY
# ──────────────────────────────────────────────────────────────────────────────

def solve_with_retry(client: AstrometryNet, file_path: Path, max_attempts: int = 3) -> fits.Header:
    for attempt in range(1, max_attempts + 1):
        dots = "." * (attempt * 6)
        print(f"  Solving{dots} (attempt {attempt}/{max_attempts}) -> ", end="", flush=True)
        try:
            wcs_header = client.solve_from_image(
                str(file_path),
                solve_timeout=180,
            )
            if wcs_header is not None:
                print("SUCCESS")
                return wcs_header
            print("Server returned empty result.")

        except Exception as e:
            print(f"API error: {e}")

        if attempt < max_attempts:
            print("  Waiting 10 s before retry...")
            time.sleep(10)
            try:
                client.login()
            except Exception:
                pass

    raise RuntimeError(
        f"Plate solving permanently failed for '{file_path.name}' "
        f"after {max_attempts} attempt(s)."
    )


# ──────────────────────────────────────────────────────────────────────────────
#   STEP 3: ESO ARCHIVE QUERY
# ──────────────────────────────────────────────────────────────────────────────

def query_eso_muse_fov(wcs: WCS, nx: int, ny: int):
    tap = vo.dal.TAPService(ESO_TAP_URL)
    
    corners_pix = np.array([[0, 0], [nx - 1, 0], [nx - 1, ny - 1], [0, ny - 1]], dtype=np.float64)
    ra_vals, dec_vals = wcs.pixel_to_world_values(corners_pix[:, 0], corners_pix[:, 1])
    
    dec_vals = np.clip(dec_vals, -90.0, 90.0)
    
    poly_pts = [f"{r:.5f}, {d:.5f}" for r, d in zip(ra_vals, dec_vals)]
    poly_str = ", ".join(poly_pts)

    print(f"  [ESO TAP] Querying for Level 3 MUSE cubes intersecting FOV polygon...", flush=True)

    adql = f"""
        SELECT dp_id, target_name, s_ra, s_dec, s_fov, t_exptime, access_url
        FROM   ivoa.ObsCore
        WHERE  instrument_name = 'MUSE'
          AND  dataproduct_type = 'cube'
          AND  calib_level = 3
          AND  INTERSECTS(s_region, POLYGON('ICRS', {poly_str})) = 1
        ORDER BY t_exptime DESC
    """

    try:
        results = tap.search(adql, maxrec=50000).to_table()
    except Exception as e:
        print(f"  [ESO TAP] Query error: {e}")
        return None

    if len(results) == 0:
        print(f"  [ESO TAP] No Level 3 MUSE cubes found in this FOV.")
        return None

    print(f"  [ESO TAP] Found {len(results)} overlapping Level 3 MUSE cube(s).")
    return results


# ──────────────────────────────────────────────────────────────────────────────
#   STEP 4: DIRECT DOWNLOAD
# ──────────────────────────────────────────────────────────────────────────────

def build_session(username=None, password=None) -> requests.Session:
    session = requests.Session()
    session.headers.update({"User-Agent": "ESO-MUSE-Pipeline/1.0"})

    if username and password:
        print("  [auth] Logging in to ESO User Portal...")
        login_url = "https://www.eso.org/sso/oidc/token"
        try:
            resp = session.post(
                login_url,
                data={
                    "grant_type": "password",
                    "client_id":  "archive-api",
                    "username":   username,
                    "password":   password,
                },
                timeout=30,
            )
            resp.raise_for_status()
            token = resp.json().get("access_token")
            if token:
                session.headers["Authorization"] = f"Bearer {token}"
                print("  [auth] ESO login successful.")
        except Exception as e:
            print(f"  [auth] ESO login failed: {e}. Proceeding anonymously.")

    return session


def download_cube_direct(dp_id: str, output_path: Path, session: requests.Session) -> Path:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = output_path.with_suffix(output_path.suffix + ".part")

    direct_url = f"https://dataportal.eso.org/dataPortal/file/{dp_id}"
    print(f"  [download] URL : {direct_url}")

    try:
        with session.get(direct_url, stream=True, timeout=DOWNLOAD_TIMEOUT) as resp:
            content_type = resp.headers.get("Content-Type", "")
            if "text/html" in content_type or "xml" in content_type:
                raise RuntimeError(
                    f"ESO returned non-binary content ({content_type}). "
                    "The cube may be proprietary — ensure ESO credentials are correct."
                )

            resp.raise_for_status()
            total   = int(resp.headers.get("Content-Length", 0))
            written = 0

            with open(temp_path, "wb") as f:
                for chunk in resp.iter_content(chunk_size=4 * 1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        written += len(chunk)
                        if total:
                            pct = written / total * 100
                            print(f"\r  [download] {written/1e6:6.1f} MB / {total/1e6:.1f} MB  ({pct:.1f}%)", end="", flush=True)

        print()
        if total > 0 and written != total:
            raise IOError(f"Size mismatch: expected {total} B, received {written} B.")

        temp_path.replace(output_path)

        try:
            with fits.open(output_path, mode="update") as hdul:
                if "SIMPLE" not in hdul[0].header:
                    hdul[0].header.insert(0, ("SIMPLE", True, "file conforms to FITS standard"))
                    hdul.flush()
                    print("  [download] Added missing SIMPLE card to primary header.")
        except Exception as e:
            print(f"  [download] Warning: Could not verify/add SIMPLE card: {e}")

        print(f"  [download] Saved: {output_path.name}  ({written/1e6:.1f} MB)")
        return output_path

    except Exception as e:
        if temp_path.exists():
            temp_path.unlink()
        raise RuntimeError(f"Download failed for {dp_id}: {e}") from e


# ──────────────────────────────────────────────────────────────────────────────
#   MAIN BATCH LOOP
# ──────────────────────────────────────────────────────────────────────────────

def run_batch(base_dir: Path, input_dir: Path, aligned_dir: Path, cube_dir: Path, manifest_dir: Path, temp_dir: Path, api_key: str, eso_user: str, eso_pass: str, max_cubes: int = None):
    for d in [base_dir, input_dir, aligned_dir, cube_dir, manifest_dir, temp_dir]:
        d.mkdir(parents=True, exist_ok=True)

    manifest_path = manifest_dir / "batch_manifest.txt"

    astrometry = AstrometryNet()
    astrometry.api_key = api_key

    dl_session = build_session(eso_user, eso_pass)

    # Exclude hidden files and directories (like .ipynb_checkpoints)
    tiffs = [p for p in input_dir.iterdir() if p.is_file() and not p.name.startswith(".") and p.suffix.lower() in [".tif", ".tiff"]]
    
    if tiffs:
        print(f"[+] Pre-converting {len(tiffs)} TIFF(s) to FITS safely...")
        for tif in tiffs:
            fits_out = input_dir / f"{tif.stem}.fits"
            if not fits_out.exists():
                try:
                    data = tifffile.imread(tif).astype(np.float32)
                    if data.ndim == 3:
                        if data.shape[0] in (3, 4):
                            data = np.mean(data, axis=0)
                        elif data.shape[-1] in (3, 4):
                            data = np.mean(data, axis=-1)
                        else:
                            data = data[0]
                    elif data.ndim > 3:
                        while data.ndim > 2:
                            data = data[0]
                    fits.PrimaryHDU(data=data).writeto(fits_out, overwrite=True)
                    print(f"  Converted: {tif.name} -> {fits_out.name}")
                except Exception as e:
                    print(f"  [!] Failed to convert {tif.name} via tifffile: {e}")

    fits_files = sorted([p for p in input_dir.iterdir() if p.is_file() and not p.name.startswith(".") and p.suffix.lower() == ".fits"])
    if not fits_files:
        print(f"[!] No FITS files found in {input_dir}. Exiting.")
        return

    print(f"[+] Found {len(fits_files)} file(s) in {input_dir}.\n")

    with open(manifest_path, "w") as log:
        log.write("Filename | Status | RA | Dec | MUSE_dp_ids\n")
        log.write("-" * 80 + "\n")

        for img_path in fits_files:
            print(f"{'='*60}")
            print(f"[1/3] Plate solving: {img_path.name}")

            file_to_solve = prepare_image_for_solving(
                img_path, temp_dir, downsample_factor=DOWNSAMPLE_FACTOR
            )

            ra = dec = None

            try:
                wcs_header = solve_with_retry(astrometry, file_to_solve)
                wcs        = WCS(wcs_header)
                shape      = getattr(wcs, "array_shape", None)
                if shape is None:
                    with fits.open(file_to_solve) as hdul:
                        shape = hdul[0].data.shape

                ra, dec = wcs.pixel_to_world_values(
                    shape[-1] / 2.0,
                    shape[-2] / 2.0,
                )
                ra, dec = float(ra), float(dec)
                print(f"  Field centre: RA={ra:.5f}  Dec={dec:.5f}")

                cd = wcs.pixel_scale_matrix
                det = np.linalg.det(cd)

                with fits.open(img_path) as hdul_orig:
                    orig_data   = hdul_orig[0].data.astype(np.float32)

                if orig_data.ndim == 3:
                    if orig_data.shape[0] in (3, 4):
                        orig_data = np.mean(orig_data, axis=0)
                    elif orig_data.shape[-1] in (3, 4):
                        orig_data = np.mean(orig_data, axis=-1)
                    else:
                        orig_data = orig_data[0]
                elif orig_data.ndim > 3:
                    while orig_data.ndim > 2:
                        orig_data = orig_data[0]

                if det < 0:
                    print("  [alignment] Parity inversion detected (mirrored). Flipping horizontally...")
                    orig_data = np.fliplr(orig_data)

                angle_deg = np.rad2deg(np.arctan2(cd[1, 0], cd[0, 0]))
                print(f"  [alignment] Detected rotation angle: {angle_deg:.2f}°")

                rotated_data = scipy.ndimage.rotate(
                    orig_data, -angle_deg, reshape=True, order=1,
                    mode='constant', cval=np.nanmedian(orig_data)
                )

                old_ny, old_nx = orig_data.shape
                new_ny, new_nx = rotated_data.shape
                old_cx, old_cy = (old_nx - 1) / 2.0, (old_ny - 1) / 2.0
                new_cx, new_cy = (new_nx - 1) / 2.0, (new_ny - 1) / 2.0

                aligned_hdu = fits.PrimaryHDU(data=rotated_data)

                for key in wcs_header.keys():
                    if key not in ['SIMPLE', 'BITPIX', 'NAXIS', 'NAXIS1', 'NAXIS2', 'EXTEND', 'END', '']:
                        try:
                            aligned_hdu.header[key] = (wcs_header[key], wcs_header.comments[key])
                        except Exception:
                            try:
                                aligned_hdu.header[key] = wcs_header[key]
                            except Exception:
                                pass

                if 'CRPIX1' in wcs_header and 'CRPIX2' in wcs_header:
                    aligned_hdu.header['CRPIX1'] = float(wcs_header['CRPIX1']) + (new_cx - old_cx)
                    aligned_hdu.header['CRPIX2'] = float(wcs_header['CRPIX2']) + (new_cy - old_cy)

                scale = np.sqrt(np.abs(det))
                aligned_hdu.header['CD1_1'] = -scale
                aligned_hdu.header['CD1_2'] = 0.0
                aligned_hdu.header['CD2_1'] = 0.0
                aligned_hdu.header['CD2_2'] = scale

                aligned_path = aligned_dir / f"aligned_{img_path.name}"
                aligned_hdu.writeto(aligned_path, overwrite=True)
                print(f"  [alignment] Saved North-up aligned raw image with WCS: {aligned_path.name}")

                print(f"\n[2/3] Querying ESO archive for FOV intersections...")
                aligned_wcs = WCS(aligned_hdu.header)
                muse_table = query_eso_muse_fov(aligned_wcs, new_nx, new_ny)

                if muse_table is not None and len(muse_table) > 0:
                    print(f"\n[3/3] Found {len(muse_table)} valid cubes. Commencing downloads...")
                    downloaded_ids = []
                    
                    for i, row in enumerate(muse_table, 1):
                        if max_cubes and i > max_cubes:
                            print(f"\n  [!] Reached payload limit of {max_cubes} cube(s). Skipping remaining.")
                            break

                        dp_id  = str(row["dp_id"])
                        target = str(row.get("target_name", "unknown")).replace(" ", "_")
                        expt   = float(row.get("t_exptime", 0.0))
                        
                        print(f"\n  ({i}/{len(muse_table)}) Target: {target} | ExpTime: {expt:.1f}s")
                        
                        safe_stem = dp_id.replace(":", "_").replace(".", "_")
                        cube_path = cube_dir / f"{safe_stem}.fits"

                        if cube_path.exists():
                            print(f"  [skip] {cube_path.name} already exists.")
                            downloaded_ids.append(dp_id)
                            continue
                            
                        try:
                            download_cube_direct(dp_id, cube_path, dl_session)
                            downloaded_ids.append(dp_id)
                        except Exception as e:
                            print(f"  [error] Failed to download {dp_id}: {e}")

                    log.write(
                        f"{img_path.name} | SUCCESS | "
                        f"{ra:.5f} | {dec:.5f} | {','.join(downloaded_ids)}\n"
                    )
                else:
                    print(f"\n[3/3] No suitable Level-3 MUSE cubes found — skipping download.")
                    log.write(
                        f"{img_path.name} | NO_CUBES | "
                        f"{ra:.5f} | {dec:.5f} | None\n"
                    )

            except Exception as e:
                print(f"\n[!] Pipeline error for {img_path.name}: {e}")
                ra_str  = f"{ra:.5f}"  if ra  is not None else "N/A"
                dec_str = f"{dec:.5f}" if dec is not None else "N/A"
                log.write(
                    f"{img_path.name} | FAILED | "
                    f"{ra_str} | {dec_str} | Error: {e}\n"
                )

            finally:
                solve_cleanup = temp_dir / f"solve_{img_path.name}"
                if solve_cleanup.exists():
                    solve_cleanup.unlink()

            print()

    print(f"[+] Batch complete. Manifest: {manifest_path}")

def main(base_dir=Path("./SOLVE"), input_dir=None, aligned_dir=None, cube_dir=None, manifest_dir=None, temp_dir=None, api_key=None, eso_user=None, eso_pass=None, max_cubes=None):
    if not api_key or api_key == "your_actual_astrometry_key_here":
        raise ValueError("Astrometry.net API key is required. Please update the configuration block or pass it as an argument.")
        
    input_dir    = input_dir if input_dir else base_dir / "INPUT"
    aligned_dir  = aligned_dir if aligned_dir else base_dir / "ALIGNED"
    cube_dir     = cube_dir if cube_dir else base_dir / "MUSE_CUBES"
    manifest_dir = manifest_dir if manifest_dir else base_dir / "manifests"
    temp_dir     = temp_dir if temp_dir else base_dir / "temp"

    print(f"Starting pipeline...")
    print(f"  Base Dir:     {base_dir.resolve()}")
    print(f"  Input Dir:    {input_dir.resolve()}")
    print(f"  Aligned Dir:  {aligned_dir.resolve()}")
    print(f"  Cubes Dir:    {cube_dir.resolve()}")
    print(f"  Manifest Dir: {manifest_dir.resolve()}")
    print(f"  Temp Dir:     {temp_dir.resolve()}")
    if max_cubes:
        print(f"  Payload Limit: {max_cubes} cube(s) per field\n")
    else:
        print(f"  Payload Limit: None (Downloading ALL matching cubes)\n")

    run_batch(
        base_dir=base_dir,
        input_dir=input_dir,
        aligned_dir=aligned_dir,
        cube_dir=cube_dir,
        manifest_dir=manifest_dir,
        temp_dir=temp_dir,
        api_key=api_key,
        eso_user=eso_user,
        eso_pass=eso_pass,
        max_cubes=max_cubes
    )

if __name__ == "__main__" or "__file__" not in globals():
    if "__file__" in globals():
        parser = argparse.ArgumentParser(description="ESO MUSE Pipeline")
        parser.add_argument("--base-dir", type=Path, default=BASE_DIR)
        parser.add_argument("--input-dir", type=Path, default=None)
        parser.add_argument("--aligned-dir", type=Path, default=None)
        parser.add_argument("--cube-dir", type=Path, default=None)
        parser.add_argument("--manifest-dir", type=Path, default=None)
        parser.add_argument("--temp-dir", type=Path, default=None)
        parser.add_argument("--api-key", type=str, default=ASTROMETRY_API_KEY)
        parser.add_argument("--eso-user", type=str, default=ESO_USERNAME)
        parser.add_argument("--eso-pass", type=str, default=ESO_PASSWORD)
        parser.add_argument("--max-cubes", type=int, default=MAX_CUBES, help="Limit number of cubes downloaded per field.")
        args = parser.parse_args()
        
        main(args.base_dir, args.input_dir, args.aligned_dir, args.cube_dir, args.manifest_dir, args.temp_dir, args.api_key, args.eso_user, args.eso_pass, args.max_cubes)
    else:
        main(BASE_DIR, INPUT_DIR, ALIGNED_DIR, CUBE_DIR, MANIFEST_DIR, TEMP_DIR, ASTROMETRY_API_KEY, ESO_USERNAME, ESO_PASSWORD, MAX_CUBES)

Starting pipeline...
  Base Dir:     C:\Users\TNTWO\menon_lab\GIT+LAB\MUSE_Download\SOLVE
  Input Dir:    C:\Users\TNTWO\menon_lab\GIT+LAB\MUSE_Download\SOLVE\INPUT
  Aligned Dir:  C:\Users\TNTWO\menon_lab\GIT+LAB\MUSE_Download\SOLVE\ALIGNED
  Cubes Dir:    C:\Users\TNTWO\menon_lab\GIT+LAB\MUSE_Download\SOLVE\MUSE_CUBES
  Manifest Dir: C:\Users\TNTWO\menon_lab\GIT+LAB\MUSE_Download\SOLVE\manifests
  Temp Dir:     C:\Users\TNTWO\menon_lab\GIT+LAB\MUSE_Download\SOLVE\TEMP
  Payload Limit: 1 cube(s) per field

[+] Pre-converting 1 TIFF(s) to FITS safely...
[+] Found 1 file(s) in SOLVE\INPUT.

[1/3] Plate solving: NGC3201_ST.fits
  Solving...... (attempt 1/3) -> Determining background stats
Finding sources
Found 46 sources


 id     x_centroid     ...     mag          daofind_mag      
--- ------------------ ... ----------- ----------------------
 42 34.849549276508036 ...  -14.688372   -0.32517720199308076
 40 48.750110355040654 ...  -14.665968    -0.5104479995449192
 29  82.66133792486379 ...   -14.59382   -0.17998816653485486
 17  72.85631936579198 ...  -14.515667    -0.3664240684498882
 19 25.461421308240567 ...  -14.405623    -0.6043243916938985
 30  72.10913857454088 ...  -14.403458 -0.0035586924950593898
 32 53.593865628046245 ...  -14.249063     -0.322422204335553
 35  57.13225811705812 ...  -14.225832    -0.5017504785495047
 24   72.8844496051538 ...  -13.948842   -0.41083969112117674
 37  3.441497971283729 ...  -13.910946   -0.49968897750350705
...                ... ...         ...                    ...
 28 106.14157697608306 ...  -12.598325    -0.1916110249084148
 25 118.77677399310491 ...  -12.581703   -0.11805555768919777
  1  44.45841021405258 ...  -12.573957   -0.17305160581506085
 34 145.

Solving......................................................................SUCCESS
  Field centre: RA=154.40904  Dec=-46.41512
  [alignment] Detected rotation angle: -88.64°


  [alignment] Saved North-up aligned raw image with WCS: aligned_NGC3201_ST.fits

[2/3] Querying ESO archive for FOV intersections...
  [ESO TAP] Querying for Level 3 MUSE cubes intersecting FOV polygon...
  [ESO TAP] Found 1 overlapping Level 3 MUSE cube(s).

[3/3] Found 1 valid cubes. Commencing downloads...

  (1/1) Target: NGC3201 | ExpTime: 12957.0s
  [download] URL : https://dataportal.eso.org/dataPortal/file/ADP.2019-12-05T10:37:27.936

  [download] Saved: ADP_2019-12-05T10_37_27_936.fits  (7532.0 MB)

[+] Batch complete. Manifest: SOLVE\manifests\batch_manifest.txt
